# Phase 1B v2 — ControlNet + img2img + LoRA

**Based on supervisor feedback:** instead of correcting colours after generation,
feed the quantized colour image as an extra input so the model can *see* the
colours directly during generation.

**Three inputs to the model:**
```
canny_final.png   → ControlNet  → constrains structure (face layout, hair, clothing)
quantized.png     → img2img     → provides colour reference (the model sees your colours)
prompt + LoRA     → SD 1.5      → drives pixel art style
```

**Key parameter:** `strength` (0.0–1.0)
- Low (0.3–0.5): output stays close to quantized.png colours, less creative
- High (0.7–0.9): model is more creative, may drift from original colours
- Start at 0.6 and tune from there.

**Required uploads:** `canny_final.png` + `quantized.png` + `stardew_pixel_lora_final.safetensors`

**Run order:** Cell 0 → Restart Runtime → Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6

In [ ]:
# Cell 0: Install
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('[WARN]', r.stderr[-400:])

print('[1/4] huggingface_hub==0.21.4')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
     'huggingface_hub==0.21.4'])

print('[2/4] diffusers / transformers / peft (--no-deps)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
     'diffusers==0.27.2', 'transformers==4.39.3', 'accelerate==0.29.3',
     'safetensors==0.4.3', 'tokenizers==0.15.2', 'peft==0.10.0',
     'einops', 'omegaconf'])

print('[3/4] scikit-learn + opencv')
run([sys.executable, '-m', 'pip', 'install', '--quiet',
     'scikit-learn', 'opencv-python-headless'])

print('[4/4] numpy==1.26.4 (pin last)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
     'numpy==1.26.4'])

import numpy
print(f'numpy: {numpy.__version__}  (must be 1.26.x)')
print('Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.')

[1/4] huggingface_hub==0.21.4
[2/4] diffusers / transformers / peft (--no-deps)
[3/4] scikit-learn + opencv
[4/4] numpy==1.26.4 (pin last)
numpy: 1.26.4  (must be 1.26.x)
Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.


In [ ]:
# Cell 1: Upload files
from google.colab import files
from PIL import Image
import io, os

print('Upload: canny_final.png + quantized.png + stardew_pixel_lora_final.safetensors')
uploaded = files.upload()

CANNY_PATH = None
QUANT_PATH = None
LORA_PATH  = None

for fname, data in uploaded.items():
    path = f'/content/{fname}'
    with open(path, 'wb') as f:
        f.write(data)
    if 'canny' in fname.lower():
        CANNY_PATH = path
        print(f'Canny edge map: {path}')
        display(Image.open(io.BytesIO(data)))
    elif 'quantized' in fname.lower():
        QUANT_PATH = path
        print(f'Quantized colour image: {path}')
        display(Image.open(io.BytesIO(data)))
    elif fname.endswith('.safetensors'):
        LORA_PATH = path
        print(f'LoRA weights: {path} ({os.path.getsize(path)/1e6:.1f} MB)')

assert CANNY_PATH, 'Missing canny_final.png'
assert QUANT_PATH, 'Missing quantized.png'
assert LORA_PATH,  'Missing .safetensors'
print('All files ready.')

Upload: canny_final.png + quantized.png + stardew_pixel_lora_final.safetensors


Saving canny_final (2).png to canny_final (2) (1).png
Saving quantized (2).png to quantized (2) (1).png
Saving stardew_pixel_lora_final (5).safetensors to stardew_pixel_lora_final (5) (1).safetensors
Canny edge map: /content/canny_final (2) (1).png


Quantized colour image: /content/quantized (2) (1).png


LoRA weights: /content/stardew_pixel_lora_final (5) (1).safetensors (16.0 MB)
All files ready.


In [ ]:
# Cell 2: CONFIG
CONFIG = {
    'base_model_id'   : 'runwayml/stable-diffusion-v1-5',
    'controlnet_id'   : 'lllyasviel/sd-controlnet-canny',
    'lora_path'       : LORA_PATH,
    'canny_path'      : CANNY_PATH,
    'quantized_path'  : QUANT_PATH,

    'prompt'          : (
        'stardew_pixel_art, 1girl, solo, portrait, '
        'white background, pure white background, simple background, '
        'upper body, facing viewer, pixel art'
    ),
    'negative_prompt' : (
        'red background, gray background, colorful background, gradient, '
        'blurry, realistic, 3d, multiple characters'
    ),

    # ControlNet: how strictly to follow the edge map structure
    'controlnet_scale': 0.8,

    # LoRA: pixel art style strength
    'lora_scale'      : 0.8,

    # img2img strength: how much the model can deviate from quantized.png
    # 0.3 = very close to original colours
    # 0.6 = balanced (recommended start)
    # 0.9 = nearly ignores the colour reference
    'strength'        : 0.6,

    'num_inference_steps': 30,
    'guidance_scale'     : 7.5,
    'seed'               : 42,

    # Post-processing pixel block size
    'pixel_size'      : 8,
}
print('Config ready.')
print(f'ControlNet scale: {CONFIG["controlnet_scale"]}  |  '
      f'LoRA scale: {CONFIG["lora_scale"]}  |  '
      f'img2img strength: {CONFIG["strength"]}')

Config ready.
ControlNet scale: 0.8  |  LoRA scale: 0.8  |  img2img strength: 0.6


In [ ]:
# Cell 3: Load pipeline + merge LoRA
# Key change from Phase 1B v1:
#   StableDiffusionControlNetPipeline     (text + edges only)
#   → StableDiffusionControlNetImg2ImgPipeline  (text + edges + colour image)

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--force-reinstall', 'numpy==1.26.4'], capture_output=True)

import torch
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetImg2ImgPipeline,
    UniPCMultistepScheduler,
)
from safetensors.torch import load_file
from PIL import Image

print('Loading ControlNet...')
controlnet = ControlNetModel.from_pretrained(
    CONFIG['controlnet_id'], torch_dtype=torch.float16)

print('Loading SD 1.5 + ControlNet img2img pipeline...')
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    CONFIG['base_model_id'],
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing()

# Merge LoRA: W_new = W_orig + lora_scale * (lora_B @ lora_A)
print('Merging LoRA weights...')
lora_weights = load_file(CONFIG['lora_path'])
unet_sd = pipe.unet.state_dict()
merged, skipped = 0, 0

for a_key in [k for k in lora_weights if k.endswith('lora_A.default.weight')]:
    b_key    = a_key.replace('lora_A', 'lora_B')
    base_key = a_key.replace('base_model.model.', '').replace('.lora_A.default.weight', '.weight')
    if b_key not in lora_weights or base_key not in unet_sd:
        skipped += 1; continue
    lora_A = lora_weights[a_key].float().cpu()
    lora_B = lora_weights[b_key].float().cpu()
    orig   = unet_sd[base_key].float().cpu()
    A_2d   = lora_A.view(lora_A.shape[0], -1)
    B_2d   = lora_B.view(lora_B.shape[0], -1)
    unet_sd[base_key] = orig + (B_2d @ A_2d).view(orig.shape) * CONFIG['lora_scale']
    merged += 1

pipe.unet.load_state_dict(unet_sd, strict=True)
pipe.unet = pipe.unet.half()
print(f'LoRA merged: {merged} weights applied, {skipped} skipped')
print('Pipeline ready.')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Loading ControlNet...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading SD 1.5 + ControlNet img2img pipeline...


You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet_img2img.StableDiffusionControlNetImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Merging LoRA weights...
LoRA merged: 160 weights applied, 0 skipped
Pipeline ready.


In [ ]:
# Cell 4: Generate
# The quantized colour image is passed as `image` (img2img init image).
# The canny edge map is passed as `control_image` (ControlNet conditioning).
# The model starts from the colour image (adding noise scaled by `strength`)
# then denoises guided by both the edges and the prompt+LoRA.

import torch
from PIL import Image
from IPython.display import display

colour_image = Image.open(CONFIG['quantized_path']).convert('RGB').resize((512,512))
canny_image  = Image.open(CONFIG['canny_path']).convert('RGB').resize((512,512))
generator    = torch.Generator(device='cuda').manual_seed(CONFIG['seed'])

print(f'Generating with strength={CONFIG["strength"]}...')
with torch.autocast('cuda'):
    result = pipe(
        prompt                        = CONFIG['prompt'],
        negative_prompt               = CONFIG['negative_prompt'],
        image                         = colour_image,   # colour reference
        control_image                 = canny_image,    # structure reference
        controlnet_conditioning_scale = CONFIG['controlnet_scale'],
        strength                      = CONFIG['strength'],
        num_inference_steps           = CONFIG['num_inference_steps'],
        guidance_scale                = CONFIG['guidance_scale'],
        generator                     = generator,
    ).images[0]

result.save('/content/generated_img2img.png')
print('Generated.')
display(result)

Generating with strength=0.6...


Generated.


In [ ]:
# Cell 5: strength sweep — find the best balance
# Low strength = faithful colours, less pixel art style
# High strength = more creative, may lose original colours

import torch
import matplotlib.pyplot as plt
from PIL import Image

strengths   = [0.3, 0.5, 0.7, 0.9]
colour_image = Image.open(CONFIG['quantized_path']).convert('RGB').resize((512,512))
canny_image  = Image.open(CONFIG['canny_path']).convert('RGB').resize((512,512))
results      = []

for s in strengths:
    print(f'  strength={s}...')
    gen = torch.Generator(device='cuda').manual_seed(CONFIG['seed'])
    with torch.autocast('cuda'):
        img = pipe(
            prompt=CONFIG['prompt'],
            negative_prompt=CONFIG['negative_prompt'],
            image=colour_image,
            control_image=canny_image,
            controlnet_conditioning_scale=CONFIG['controlnet_scale'],
            strength=s,
            num_inference_steps=CONFIG['num_inference_steps'],
            guidance_scale=CONFIG['guidance_scale'],
            generator=gen,
        ).images[0]
    results.append((s, img))

fig, axes = plt.subplots(1, len(strengths)+1, figsize=(5*(len(strengths)+1), 5))
axes[0].imshow(colour_image)
axes[0].set_title('Colour reference\n(quantized.png)')
axes[0].axis('off')
for ax, (s, img) in zip(axes[1:], results):
    ax.imshow(img)
    ax.set_title(f'strength={s}')
    ax.axis('off')
plt.suptitle('img2img strength sweep\n(lower = closer to original colours)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/strength_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pick the strength that best balances colour fidelity and pixel art style.')

  strength=0.3...


  strength=0.5...


  strength=0.7...


  strength=0.9...


Pick the strength that best balances colour fidelity and pixel art style.


In [ ]:
# Cell 6: Final comparison — v1 (post-process remap) vs v2 (img2img)
# Use this to show your supervisor the difference between the two approaches.

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(Image.open(CONFIG['quantized_path']))
axes[0].set_title('Colour reference input\n(quantized.png from Phase 1A)')
axes[0].axis('off')

if os.path.exists('/content/final_output.png'):
    axes[1].imshow(mpimg.imread('/content/final_output.png'))
    axes[1].set_title('v1: ControlNet + post-process remap\n(colour corrected after generation)')
else:
    axes[1].text(0.5, 0.5, 'Run Phase 1B v1 first\nto compare', ha='center', va='center')
    axes[1].set_title('v1 output (not available)')
axes[1].axis('off')

axes[2].imshow(mpimg.imread('/content/generated_img2img.png'))
axes[2].set_title('v2: ControlNet + img2img\n(colour fed in during generation)')
axes[2].axis('off')

plt.suptitle(
    'Approach comparison\n'
    'v1: post-process colour remap  |  v2: img2img colour conditioning (supervisor suggestion)',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/approach_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

from google.colab import files
for path in ['/content/generated_img2img.png',
             '/content/strength_sweep.png',
             '/content/approach_comparison.png']:
    if os.path.exists(path):
        files.download(path)